# Hydrogen project data preparation

This notebook builds the harmonised IEA project-year panel and prepares the discrete-time complementary log-log dataset.

## Pipeline conventions

- A project that disappears from a later IEA vintage is classified as cancelled.
- Sensitivity convention for the 2026 IEA statuses: `Dormant` and `On hold` inherit the project's most recent prior non-terminal development stage (`Concept`, `Feasibility study`, or `FID/Construction`). They therefore remain in the risk set and are right-censored if still dormant/on hold in the final vintage. The original source label is retained in `status_original`, and `status_prior_stage_carried` flags these rows.
- Project characteristics are carried forward when missing project-year rows are completed.
- All rows in an unfinished final status spell are marked as `right_censored`.
- Project density is calculated separately by data vintage, excludes the focal project, and uses geodesic distances.
- Absorbing project states are propagated through subsequent project-year rows.
- Country codes are standardised before external-data joins.
- Multi-country projects receive equally weighted country-level controls and a region classification based on the constituent countries.
- The unstandardised time-in-stage variable is stored as `prev_time_in_status_raw` for prediction figures expressed in calendar years.
- `cloglog_data_processed.rds` is the model-ready dataset, `cloglog_data_full.rds` is the complete transition dataset, and `master_data.rds` is the complete project-year panel.
`Governance_Score` and `Ease_of_doing_business` are both retained and standardised in the model-ready data. The governance control used in estimation is selected in the master pipeline notebook.


## 1. Setup


In [1]:
# ------------------------------------------------------------------------------
# Working directory and packages
# ------------------------------------------------------------------------------

# Use the same project/data directory in standalone and master-run execution.
# With H2_PROJECT_DIR unset, run from the folder containing Data/ and the RDS files.
project_dir <- Sys.getenv("H2_PROJECT_DIR", unset = getwd())
project_dir <- normalizePath(project_dir, winslash = "/", mustWork = TRUE)
setwd(project_dir)

library(dplyr)
library(tidyr)
library(purrr)
library(stringr)
library(sf)

# ------------------------------------------------------------------------------
# Figure 2 model-data options
# ------------------------------------------------------------------------------

# Set to FALSE to keep transformed numeric variables in their original units.
standardize_numeric <- TRUE

# Preserve the full transition dataset and separately save the Figure 2-ready
# dataset consumed by the sequential Figure 2 notebook.
cloglog_full_path <- "cloglog_data_full.rds"
cloglog_figure2_path <- "cloglog_data_processed.rds"

# Audit identifiers and counts without modifying or deduplicating model data.
# Empty identifiers use the same definition as Figure 1 / ST1-2 (reference == "").
# Whitespace-only identifiers are flagged separately and are not silently recoded.
audit_project_year_keys <- function(data, dataset) {
  stopifnot(all(c("reference", "year") %in% names(data)))
  ref <- as.character(data$reference)
  yr <- suppressWarnings(as.integer(as.character(data$year)))
  indexed <- tibble::tibble(
    audit_row = seq_len(nrow(data)), reference = ref, year = yr,
    exclusion_reason = dplyr::case_when(
      is.na(ref) ~ "Missing reference",
      ref == "" ~ "Empty reference",
      is.na(yr) ~ "Missing or invalid year",
      TRUE ~ "Valid key"
    ),
    whitespace_only_reference = !is.na(ref) & ref != "" & trimws(ref) == ""
  )
  valid <- indexed %>% dplyr::filter(exclusion_reason == "Valid key")
  key_counts <- valid %>% dplyr::count(reference, year, name = "rows_per_key")
  duplicate_key_details <- key_counts %>% dplyr::filter(rows_per_key > 1L)
  payload_vars <- setdiff(names(data), c(".iea_source_file", ".iea_source_excel_row"))
  duplicate_key_details$distinct_payloads <- integer(nrow(duplicate_key_details))
  duplicate_key_details$conflicting_fields <- character(nrow(duplicate_key_details))
  for (i in seq_len(nrow(duplicate_key_details))) {
    rows <- valid$audit_row[
      valid$reference == duplicate_key_details$reference[i] & valid$year == duplicate_key_details$year[i]
    ]
    payload <- data[rows, payload_vars, drop = FALSE]
    duplicate_key_details$distinct_payloads[i] <- nrow(dplyr::distinct(payload))
    varying <- vapply(payload, function(x) dplyr::n_distinct(x) > 1L, logical(1))
    duplicate_key_details$conflicting_fields[i] <- paste(names(payload)[varying], collapse = "; ")
  }
  indexed <- indexed %>% dplyr::left_join(key_counts, by = c("reference", "year"))
  indexed$extra_duplicate_row <- FALSE
  indexed$extra_duplicate_row[valid$audit_row] <- duplicated(valid[c("reference", "year")])
  indexed$observed_row <- if ("observed_row" %in% names(data)) {
    !is.na(data$observed_row) & as.logical(data$observed_row)
  } else rep(TRUE, nrow(data))
  key_observation <- indexed %>%
    dplyr::filter(exclusion_reason == "Valid key") %>%
    dplyr::group_by(reference, year) %>%
    dplyr::summarise(has_observed_row = any(observed_row), .groups = "drop")
  summary <- tibble::tibble(
    dataset = dataset,
    total_rows = nrow(data),
    missing_reference_rows = sum(indexed$exclusion_reason == "Missing reference"),
    empty_reference_rows = sum(indexed$exclusion_reason == "Empty reference"),
    missing_year_rows_with_valid_reference = sum(indexed$exclusion_reason == "Missing or invalid year"),
    valid_key_rows = nrow(valid),
    distinct_valid_project_years = nrow(key_counts),
    valid_projects = dplyr::n_distinct(valid$reference),
    duplicate_keys = nrow(duplicate_key_details),
    extra_duplicate_rows = sum(indexed$extra_duplicate_row),
    conflicting_duplicate_keys = sum(duplicate_key_details$distinct_payloads > 1L),
    whitespace_only_reference_rows = sum(indexed$whitespace_only_reference),
    observed_rows = sum(indexed$observed_row),
    reconstructed_rows = sum(!indexed$observed_row),
    project_years_with_observed_record = sum(key_observation$has_observed_row),
    project_years_only_reconstructed = sum(!key_observation$has_observed_row)
  )
  stopifnot(
    summary$total_rows == summary$missing_reference_rows + summary$empty_reference_rows +
      summary$missing_year_rows_with_valid_reference + summary$extra_duplicate_rows +
      summary$distinct_valid_project_years,
    summary$distinct_valid_project_years == summary$project_years_with_observed_record +
      summary$project_years_only_reconstructed
  )
  by_year <- indexed %>% dplyr::group_by(year) %>% dplyr::summarise(
    total_rows = dplyr::n(),
    invalid_key_rows = sum(exclusion_reason != "Valid key"),
    extra_duplicate_rows = sum(extra_duplicate_row),
    distinct_valid_project_years = sum(exclusion_reason == "Valid key" & !extra_duplicate_row),
    observed_rows = sum(observed_row), reconstructed_rows = sum(!observed_row),
    .groups = "drop"
  )
  flagged <- indexed %>% dplyr::filter(
    exclusion_reason != "Valid key" | rows_per_key > 1L | whitespace_only_reference
  )
  # Include original attributes and workbook/Excel row provenance for investigation.
  flagged_rows <- dplyr::bind_cols(
    flagged,
    data[flagged$audit_row, setdiff(names(data), names(flagged)), drop = FALSE]
  )
  list(summary = summary, by_year = by_year, duplicate_keys = duplicate_key_details,
       flagged_rows = flagged_rows)
}

write_project_year_audit <- function(audit, prefix) {
  for (part in names(audit)) {
    export <- as.data.frame(audit[[part]])
    # CSVs retain readable representations of any multi-country list columns.
    for (column in names(export)) {
      if (is.list(export[[column]])) {
        export[[column]] <- vapply(export[[column]], function(x) paste(x, collapse = "; "), character(1))
      }
    }
    utils::write.csv(export, paste0(prefix, "_", part, ".csv"), row.names = FALSE, na = "")
  }
  print(audit$summary, width = Inf)
  print(audit$by_year, n = Inf, width = Inf)
  if (nrow(audit$duplicate_keys) > 0L) print(audit$duplicate_keys, n = Inf, width = Inf)
  invisible(audit)
}



Warning message:
“package ‘dplyr’ was built under R version 4.5.3”



Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Warning message:
“package ‘purrr’ was built under R version 4.5.3”


Linking to GEOS 3.14.1, GDAL 3.11.5, PROJ 9.7.0; sf_use_s2() is TRUE



## 2. Country-code and regional utilities


In [2]:
# ------------------------------------------------------------------------------
# Country-code standardisation
# ------------------------------------------------------------------------------

country_code_aliases <- c(
  "POR" = "PRT",
  "ROM" = "ROU",
  "UK"  = "GBR"
)

split_country_codes <- function(x) {
  lapply(as.character(x), function(value) {
    if (is.na(value) || stringr::str_squish(value) == "") {
      return(character(0))
    }

    value <- stringr::str_to_upper(value)
    tokens <- unlist(strsplit(value, "[^A-Z0-9]+"), use.names = FALSE)
    tokens <- tokens[tokens != ""]

    tokens <- ifelse(
      tokens %in% names(country_code_aliases),
      unname(country_code_aliases[tokens]),
      tokens
    )

    tokens <- tokens[nchar(tokens) == 3 | tokens == "EU"]
    unique(tokens)
  })
}

standardize_country_string <- function(x) {
  vapply(
    split_country_codes(x),
    function(codes) {
      if (length(codes) == 0) {
        return(NA_character_)
      }

      paste(codes, collapse = " ")
    },
    character(1)
  )
}


# ------------------------------------------------------------------------------
# Add standardised country structure
# ------------------------------------------------------------------------------

add_country_structure <- function(data, country_var = "country") {
  stopifnot(country_var %in% names(data))

  country_codes <- split_country_codes(data[[country_var]])

  data[[country_var]] <- vapply(
    country_codes,
    function(codes) {
      if (length(codes) == 0) {
        return(NA_character_)
      }

      paste(codes, collapse = " ")
    },
    character(1)
  )

  data$country_primary <- vapply(
    country_codes,
    function(codes) {
      if (length(codes) == 0) {
        return(NA_character_)
      }

      codes[1]
    },
    character(1)
  )

  data$n_countries <- lengths(country_codes)
  data$multi_country <- as.integer(data$n_countries > 1)

  data
}


# ------------------------------------------------------------------------------
# Broad regional classification
# ------------------------------------------------------------------------------

europe_iso3 <- c(
  "ALB", "AND", "AUT", "BEL", "BGR", "BIH", "BLR",
  "CHE", "CYP", "CZE", "DEU", "DNK", "ESP", "EST",
  "FIN", "FRA", "GBR", "GRC", "HRV", "HUN", "IRL",
  "ISL", "ITA", "LIE", "LTU", "LUX", "LVA", "MCO",
  "MDA", "MKD", "MLT", "MNE", "NLD", "NOR", "POL",
  "PRT", "ROU", "RUS", "SMR", "SRB", "SVK", "SVN",
  "SWE", "TUR", "UKR", "VAT", "XKX"
)

us_canada_iso3 <- c(
  "USA", "CAN"
)

latin_america_iso3 <- c(
  "MEX", "BLZ", "CRI", "GTM", "HND", "NIC", "PAN", "SLV",
  "ARG", "BOL", "BRA", "CHL", "COL", "ECU", "GUY", "PRY",
  "PER", "SUR", "URY", "VEN", "ATG", "BHS", "BRB", "CUB",
  "DMA", "DOM", "GRD", "HTI", "JAM", "KNA", "LCA", "VCT",
  "TTO", "GUF"
)

asia_pacific_iso3 <- c(
  "KAZ", "KGZ", "TJK", "TKM", "UZB",
  "AFG", "BGD", "BTN", "IND", "LKA", "MDV", "NPL", "PAK",
  "CHN", "HKG", "JPN", "KOR", "MAC", "MNG", "PRK", "TWN",
  "BRN", "IDN", "KHM", "LAO", "MMR", "MYS", "PHL", "SGP",
  "THA", "TLS", "VNM", "AUS", "COK", "FJI", "FSM", "KIR",
  "MHL", "NIU", "NRU", "NZL", "PLW", "PNG", "SLB", "TON",
  "TUV", "VUT", "WSM"
)

africa_middle_east_iso3 <- c(
  # North Africa
  "DZA", "EGY", "LBY", "MAR", "MRT", "SDN", "TUN",

  # Middle East
  "ARE", "BHR", "IRN", "IRQ", "ISR", "JOR", "KWT", "LBN",
  "OMN", "PSE", "QAT", "SAU", "SYR",  "YEM",

  # Sub-Saharan Africa
  "AGO", "BEN", "BFA", "BWA", "BDI", "CMR", "CPV", "CAF",
  "TCD", "COM", "COD", "COG", "CIV", "DJI", "ERI", "SWZ",
  "ETH", "GAB", "GHA", "GIN", "GMB", "GNB", "GNQ", "KEN",
  "LSO", "LBR", "MDG", "MWI", "MLI", "MUS", "MOZ", "NAM",
  "NER", "NGA", "RWA", "SEN", "SLE", "SOM", "SSD", "STP",
  "SYC", "TGO", "TZA", "UGA", "ZAF", "ZMB", "ZWE"
)

country_code_to_region <- function(code) {
  dplyr::case_when(
    code == "EU" ~ "Europe",
    code %in% europe_iso3 ~ "Europe",
    code %in% us_canada_iso3 ~ "US + Canada",
    code %in% latin_america_iso3 ~ "Latin America",
    code %in% asia_pacific_iso3 ~ "Asia Pacific",
    code %in% africa_middle_east_iso3 ~ "Africa + Middle East",
    TRUE ~ NA_character_
  )
}

assign_region_from_country <- function(x) {
  vapply(
    split_country_codes(x),
    function(codes) {
      if (length(codes) == 0) {
        return(NA_character_)
      }

      project_regions <- unique(country_code_to_region(codes))
      project_regions <- project_regions[!is.na(project_regions)]

      if (length(project_regions) == 1) {
        return(project_regions)
      }

      # Cross-region projects are assigned to "Others" for a single-valued
      # regional classification.
      "Others"
    },
    character(1)
  )
}


## 3. Read and harmonise the annual IEA project files


In [3]:
# ------------------------------------------------------------------------------
# Read IEA project data
# ------------------------------------------------------------------------------

read_iea_data <- function(years = 2021:2026) {
  files <- paste0(
    "Data/H2_IEA_",
    substr(years, 3, 4),
    "_cleaned_v3.xlsx"
  )

  clean_names <- function(x) {
    x <- as.character(x)
    x <- trimws(x)
    x <- tolower(x)
    x <- gsub("normalised", "normalized", x)
    x <- gsub("decomission", "decommission", x)
    x <- gsub("references 2", "references.2", x)
    x <- gsub("\\s+", ".", x)
    x <- gsub("[^a-z0-9.]+", ".", x)
    x <- gsub("\\.+", ".", x)
    x <- gsub("^\\.|\\.$", "", x)

    dplyr::recode(
      x,
      "ref" = "reference",
      "project.name" = "name",
      "date.online" = "date.online",
      "decommission.date" = "date.decommissioned",
      "technology.details" = "tech.comments",
      "technology.electricity" = "electricity.type",
      "technology.electricity.details" = "elec.type.dedicated",
      "announced.size" = "size.announced",
      "capacity.mwel" = "cap.mwel",
      "capacity.nm.h2.h" = "capacity.nm.h.h",
      "capacity.kt.h2.y" = "capacity.kt.h2.y",
      "capacity.t.co2.captured.y" = "capacity.t.co.captured.y",
      "iea.zero.carbon.estimated.normalized.capacity.nm.h2.hour" =
        "iea.zero.carbon.estimated.normalized.capacity.nm.h.hour",
      "enduse.iron.steel" = "enduse.ironsteel",
      "enduse.other.ind" = "enduse.otherind",
      "enduse.grid.inj" = "enduse.gridinj",
      "enduse.domestic.heat" = "enduse.domesticheat",
      "enduse.ch4.grid.inj" = "enduse.ch4gridinj",
      "enduse.ch4.mobility" = "enduse.ch4mobility",
      "refs" = "references",
      "refs.quality.check" = "references.quality.check",
      "refs.quality.checked" = "references.quality.check",
      "capacity.factor" = "lowe.cf",
      "technology.aggregate" = "technology.aggregate",
      .default = x
    )
  }

  is_unresolved_header <- function(x) {
    is.na(x) |
      x == "" |
      grepl("^column[0-9]+$", x) |
      grepl("^dummy[._]?[0-9]+$", x)
  }

  build_header <- function(header_2, header_3, header_4) {
    h2 <- clean_names(header_2)
    h3 <- clean_names(header_3)
    h4 <- clean_names(header_4)

    # Header priority is row 4, then row 3, then row 2. The unresolved mask is
    # recalculated after each fallback so a valid row-3 name is not overwritten.
    out <- h4

    unresolved <- is_unresolved_header(out)
    valid_h3 <- !is_unresolved_header(h3)
    out[unresolved & valid_h3] <- h3[unresolved & valid_h3]

    unresolved <- is_unresolved_header(out)
    valid_h2 <- !is_unresolved_header(h2)
    out[unresolved & valid_h2] <- h2[unresolved & valid_h2]

    unresolved <- is_unresolved_header(out)
    out[unresolved] <- paste0("unnamed.col.", which(unresolved))

    out <- clean_names(out)

    out <- dplyr::recode(
      out,
      "country.iso.3" = "country",
      "geographical.details.decimal.degrees.lat.n.is.positive.long.e.is.positive" =
        "location",
      .default = out
    )

    make.unique(out, sep = ".")
  }

  missing_files <- files[!file.exists(files)]

  if (length(missing_files) > 0) {
    stop(
      "Missing input files: ",
      paste(missing_files, collapse = ", ")
    )
  }

  data_all <- lapply(seq_along(years), function(i) {
    year <- years[i]

    df_raw <- readxl::read_excel(
      files[i],
      sheet = "Projects",
      col_names = FALSE,
      range = cellranger::cell_limits(c(1, 1), c(NA, NA))
    )

    header_names <- build_header(
      header_2 = unlist(df_raw[2, ]),
      header_3 = unlist(df_raw[3, ]),
      header_4 = unlist(df_raw[4, ])
    )

    df <- df_raw[-(1:4), , drop = FALSE]
    retained_excel_rows <- which(rowSums(!is.na(df)) > 0)
    df <- df[retained_excel_rows, , drop = FALSE]
    names(df) <- header_names
    # Source provenance is retained solely for the sample-count audit.
    df$.iea_source_file <- basename(files[i])
    df$.iea_source_excel_row <- retained_excel_rows + 4L

    df <- df %>%
      dplyr::mutate(
        dplyr::across(
          dplyr::any_of(c(
            "date.online",
            "date.decommissioned",
            "cap.mwel",
            "capacity.nm.h.h",
            "capacity.kt.h2.y",
            "capacity.t.co.captured.y",
            "iea.zero.carbon.estimated.normalized.capacity.nm.h.hour",
            "latitude",
            "longitude",
            "lowe.cf"
          )),
          ~ suppressWarnings(as.numeric(.x))
        ),
        dplyr::across(
          dplyr::any_of("status"),
          ~ dplyr::case_when(
            trimws(as.character(.x)) == "Decommisioned" ~ "Decommissioned",
            TRUE ~ as.character(.x)
          )
        ),
        year = year,
        version = paste0("v", year)
      )

    if ("country" %in% names(df)) {
      df <- df %>%
        dplyr::mutate(
          country = standardize_country_string(country)
        )
    }

    df
  })

  enduse_vars <- c(
    "enduse.refining",
    "enduse.ammonia",
    "enduse.methanol",
    "enduse.ironsteel",
    "enduse.otherind",
    "enduse.mobility",
    "enduse.power",
    "enduse.gridinj",
    "enduse.chp",
    "enduse.domesticheat",
    "enduse.biofuels",
    "enduse.synfuels",
    "enduse.ch4gridinj",
    "enduse.ch4mobility"
  )

  dplyr::bind_rows(data_all) %>%
    dplyr::mutate(
      dplyr::across(
        dplyr::any_of(enduse_vars),
        ~ dplyr::if_else(
          is.na(.x),
          0,
          suppressWarnings(as.numeric(.x))
        )
      )
    )
}

# ------------------------------------------------------------------------------
# Initial IEA-data diagnostics
# ------------------------------------------------------------------------------

iea_data <- read_iea_data()

# Audit raw keys before any history completion, status correction or lagging.
raw_sample_count_audit <- audit_project_year_keys(iea_data, "Raw IEA input")
write_project_year_audit(raw_sample_count_audit, "sample_count_raw")

iea_variables_by_year <- iea_data %>%
  dplyr::group_by(year) %>%
  dplyr::summarise(
    dplyr::across(
      .cols = dplyr::everything(),
      .fns = ~ sum(!is.na(.x)),
      .names = "{.col}"
    ),
    .groups = "drop"
  ) %>%
  tidyr::pivot_longer(
    cols = -year,
    names_to = "variable",
    values_to = "n_non_missing"
  ) %>%
  dplyr::arrange(year, variable)

iea_missing_pct_table <- iea_data %>%
  dplyr::group_by(year) %>%
  dplyr::summarise(
    dplyr::across(
      .cols = -version,
      .fns = ~ round(mean(is.na(.x)) * 100, 2)
    ),
    .groups = "drop"
  ) %>%
  tidyr::pivot_longer(
    cols = -year,
    names_to = "variable",
    values_to = "missing_pct"
  ) %>%
  tidyr::pivot_wider(
    names_from = year,
    values_from = missing_pct
  ) %>%
  dplyr::arrange(variable)

print(iea_missing_pct_table, n = Inf)


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`
• `` -> `...36`
• `` -> `...37`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`
• `` -> `...36`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`
• `` -> `...36`
• `` -> `...37`
• `` -> `...38`
• `` -> `...39`
• `` -> `...40`
• `` -> `...41`
• `` -> `...42`
• `` -> `...43`
• `` -> `...44`


# A tibble: 1 × 16
  dataset       total_rows missing_reference_rows empty_reference_rows
  <chr>              <int>                  <int>                <int>
1 Raw IEA input      12478                     30                    0
  missing_year_rows_with_valid_reference valid_key_rows
                                   <int>          <int>
1                                      0          12448
  distinct_valid_project_years valid_projects duplicate_keys
                         <int>          <int>          <int>
1                        12448           3200              0
  extra_duplicate_rows conflicting_duplicate_keys whitespace_only_reference_rows
                 <int>                      <int>                          <int>
1                    0                          0                              0
  observed_rows reconstructed_rows project_years_with_observed_record
          <int>              <int>                              <int>
1         12478                  0

# A tibble: 53 × 7
   variable                            `2021` `2022` `2023` `2024` `2025` `2026`
   <chr>                                <dbl>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>
 1 .iea_source_excel_row                 0      0      0      0      0      0   
 2 .iea_source_file                      0      0      0      0      0      0   
 3 cap.mwel                             25.0   24.5   19.7   17.5   16.4   22.1 
 4 capacity.kt.h2.y                     19.1   18.1   13.8   12.0   11.9   19.6 
 5 capacity.nm.h.h                      19.4   18.6   13.9   12.5   12.4   20.0 
 6 capacity.t.co.captured.y             95.4   95.6   95.4   95.9   95.8   96.4 
 7 comments                            100    100    100    100    100     89.6 
 8 country                               0.2    0.27   0.35   0.29   0.27   0.27
 9 date.decommissioned                  89.9   93.1   94.9   95.3   95.6   96.1 
10 date.online                          22.2   26.2   22.1   23.0   30.1   40.8 
11 elec.t

## 4. Read country-level external variables


In [4]:
# ------------------------------------------------------------------------------
# Read external country variables
# ------------------------------------------------------------------------------

read_external_vars <- function(path = "Data/") {
  files <- c(
    "Anhydrous Ammonia Export",
    "Competitive Industrial Performance",
    "Country Complexity Ranking",
    "Country Risk Spread",
    "Ease of doing business",
    "Freshwater Withdrawal",
    "Governance Score",
    "RISE"
  )

  mean_or_na <- function(x) {
    if (all(is.na(x))) {
      return(NA_real_)
    }

    mean(x, na.rm = TRUE)
  }

  read_single_file <- function(file_name) {
    file_path <- file.path(path, paste0(file_name, ".xlsx"))

    df <- readxl::read_excel(
      file_path,
      sheet = "Output",
      range = "B6:D10000"
    ) %>%
      dplyr::select(country = 1, value = 3) %>%
      dplyr::filter(!is.na(country)) %>%
      dplyr::mutate(
        country = standardize_country_string(country),
        value = suppressWarnings(as.numeric(value))
      ) %>%
      dplyr::filter(!is.na(country)) %>%
      dplyr::group_by(country) %>%
      dplyr::summarise(
        value = mean_or_na(value),
        .groups = "drop"
      )

    names(df)[2] <- file_name
    df
  }

  df_list <- purrr::map(files, read_single_file)
  external_variables <- purrr::reduce(
    df_list,
    dplyr::full_join,
    by = "country"
  )

  if (anyDuplicated(external_variables$country) > 0) {
    stop("External country table contains duplicate country codes after cleaning.")
  }

  external_variables
}

join_external_variables_multicountry <- function(
    data,
    external_variables,
    country_var = "country"
) {
  stopifnot(country_var %in% names(data))
  stopifnot("country" %in% names(external_variables))

  indicator_vars <- setdiff(names(external_variables), "country")
  overlapping_vars <- intersect(indicator_vars, names(data))

  if (length(overlapping_vars) > 0) {
    stop(
      "External-variable columns already exist in the project data: ",
      paste(overlapping_vars, collapse = ", ")
    )
  }

  data_indexed <- data %>%
    dplyr::mutate(.row_id = dplyr::row_number())

  country_long <- data_indexed %>%
    dplyr::transmute(
      .row_id,
      .country_code = split_country_codes(.data[[country_var]])
    ) %>%
    tidyr::unnest_longer(.country_code) %>%
    dplyr::filter(
      !is.na(.country_code),
      .country_code != "EU"
    )

  if (nrow(country_long) == 0) {
    for (indicator_var in indicator_vars) {
      data_indexed[[indicator_var]] <- NA_real_
    }

    return(data_indexed %>% dplyr::select(-.row_id))
  }

  country_controls <- country_long %>%
    dplyr::left_join(
      external_variables,
      by = c(".country_code" = "country")
    ) %>%
    dplyr::group_by(.row_id) %>%
    dplyr::summarise(
      dplyr::across(
        dplyr::all_of(indicator_vars),
        ~ if (all(is.na(.x))) NA_real_ else mean(.x, na.rm = TRUE)
      ),
      .groups = "drop"
    )

  unmatched_codes <- country_long %>%
    dplyr::anti_join(
      external_variables %>% dplyr::select(country),
      by = c(".country_code" = "country")
    ) %>%
    dplyr::distinct(.country_code) %>%
    dplyr::arrange(.country_code)

  if (nrow(unmatched_codes) > 0) {
    message(
      "Country codes without external-variable matches: ",
      paste(unmatched_codes$.country_code, collapse = ", ")
    )
  }

  data_indexed %>%
    dplyr::left_join(country_controls, by = ".row_id") %>%
    dplyr::select(-.row_id)
}


## 5. Add raster-based geographic density variables


## 6. Shared analysis variables and vintage-specific project density


In [5]:
# ------------------------------------------------------------------------------
# Shared helper functions
# ------------------------------------------------------------------------------

assign_market_readiness <- function(phase_row) {
  dplyr::case_when(
    is.na(phase_row) ~ "Unknown",
    phase_row == "High" ~ "High",
    phase_row == "Medium" ~ "Medium",
    phase_row == "Low" ~ "Low",
    TRUE ~ "Unknown"
  )
}

resolve_var <- function(data, var_name) {
  if (var_name %in% names(data)) {
    return(var_name)
  }

  matches <- names(data)[tolower(names(data)) == tolower(var_name)]

  if (length(matches) == 1) {
    return(matches)
  }

  stop("Variable not found or ambiguous: ", var_name)
}

# ------------------------------------------------------------------------------
# Vintage-specific, leave-one-out project density
# ------------------------------------------------------------------------------

add_project_kde_50km <- function(
    data,
    lon_var = "longitude",
    lat_var = "latitude",
    project_id_var = "reference",
    bandwidth_m = 150000,
    raster_resolution_m = 10000,
    target_crs = "EPSG:3035",
    output_var = "project_kde_50km",
    year_var = "year",
    observed_var = "observed_row",
    truncate_at = 3
) {
  lon_var <- resolve_var(data, lon_var)
  lat_var <- resolve_var(data, lat_var)
  year_var <- resolve_var(data, year_var)

  if (project_id_var %in% names(data)) {
    project_id_var <- resolve_var(data, project_id_var)
  } else {
    project_id_var <- NULL
  }

  # The function accepts raster_resolution_m and target_crs. Density is
  # calculated from geodesic distances and does not require a raster
  # resolution or a single projected CRS.
  invisible(raster_resolution_m)
  invisible(target_crs)

  data_indexed <- data %>%
    dplyr::mutate(
      .row_id = dplyr::row_number(),
      .lon_num = suppressWarnings(
        as.numeric(as.character(.data[[lon_var]]))
      ),
      .lat_num = suppressWarnings(
        as.numeric(as.character(.data[[lat_var]]))
      ),
      .valid_coordinates = !is.na(.lon_num) &
        !is.na(.lat_num) &
        dplyr::between(.lon_num, -180, 180) &
        dplyr::between(.lat_num, -90, 90)
    )

  if (!is.null(project_id_var)) {
    data_indexed <- data_indexed %>%
      dplyr::mutate(
        .project_id = as.character(.data[[project_id_var]])
      )
  } else {
    data_indexed <- data_indexed %>%
      dplyr::mutate(.project_id = NA_character_)
  }

  if (observed_var %in% names(data_indexed)) {
    data_indexed <- data_indexed %>%
      dplyr::mutate(
        .density_source_row = tidyr::replace_na(
          as.logical(.data[[observed_var]]),
          FALSE
        )
      )
  } else {
    data_indexed <- data_indexed %>%
      dplyr::mutate(.density_source_row = TRUE)
  }

  data_indexed[[output_var]] <- NA_real_

  old_s2 <- sf::sf_use_s2()
  on.exit(sf::sf_use_s2(old_s2), add = TRUE)
  sf::sf_use_s2(TRUE)

  bandwidth_km <- bandwidth_m / 1000
  truncation_mass <- 1 - exp(-0.5 * truncate_at^2)
  density_normalizer <- 2 * pi * bandwidth_km^2 * truncation_mass
  search_distance <- units::set_units(
    bandwidth_m * truncate_at,
    "m"
  )

  vintage_years <- sort(unique(data_indexed[[year_var]]))
  vintage_years <- vintage_years[!is.na(vintage_years)]

  for (vintage_year in vintage_years) {
    vintage_rows <- data_indexed %>%
      dplyr::filter(.data[[year_var]] == vintage_year)

    evaluation_rows <- vintage_rows %>%
      dplyr::filter(.valid_coordinates)

    if (nrow(evaluation_rows) == 0) {
      next
    }

    source_rows <- vintage_rows %>%
      dplyr::filter(
        .density_source_row,
        .valid_coordinates
      ) %>%
      dplyr::mutate(
        .source_id = dplyr::if_else(
          !is.na(.project_id) & .project_id != "",
          .project_id,
          paste0("__row_", .row_id)
        )
      ) %>%
      dplyr::arrange(.row_id) %>%
      dplyr::distinct(.source_id, .keep_all = TRUE)

    if (nrow(source_rows) == 0) {
      data_indexed[[output_var]][evaluation_rows$.row_id] <- 0
      next
    }

    evaluation_points <- evaluation_rows %>%
      sf::st_as_sf(
        coords = c(".lon_num", ".lat_num"),
        crs = 4326,
        remove = FALSE
      )

    source_points <- source_rows %>%
      sf::st_as_sf(
        coords = c(".lon_num", ".lat_num"),
        crs = 4326,
        remove = FALSE
      )

    nearby_sources <- sf::st_is_within_distance(
      evaluation_points,
      source_points,
      dist = search_distance
    )

    source_project_ids <- source_rows$.project_id
    evaluation_project_ids <- evaluation_rows$.project_id

    vintage_density <- vapply(
      seq_len(nrow(evaluation_points)),
      function(i) {
        candidate_indices <- nearby_sources[[i]]

        if (length(candidate_indices) == 0) {
          return(0)
        }

        focal_project_id <- evaluation_project_ids[i]

        if (!is.na(focal_project_id) && focal_project_id != "") {
          candidate_indices <- candidate_indices[
            is.na(source_project_ids[candidate_indices]) |
              source_project_ids[candidate_indices] != focal_project_id
          ]
        }

        if (length(candidate_indices) == 0) {
          return(0)
        }

        distances_m <- as.numeric(
          sf::st_distance(
            evaluation_points[i, ],
            source_points[candidate_indices, ],
            by_element = FALSE
          )
        )

        kernel_weights <- exp(
          -0.5 * (distances_m / bandwidth_m)^2
        )

        sum(kernel_weights, na.rm = TRUE) / density_normalizer
      },
      numeric(1)
    )

    data_indexed[[output_var]][evaluation_rows$.row_id] <- vintage_density
  }

  data_indexed %>%
    dplyr::select(
      -.row_id,
      -.lon_num,
      -.lat_num,
      -.valid_coordinates,
      -.project_id,
      -.density_source_row
    )
}

# ------------------------------------------------------------------------------
# Shared variables used by the cloglog and logit workflows
# ------------------------------------------------------------------------------

prepare_shared_analysis_data <- function(
    data,
    lon_var = "longitude",
    lat_var = "latitude",
    project_id_var = "reference",
    add_project_kde = TRUE,
    project_kde_bandwidth_m = 50000,
    project_kde_resolution_m = 10000
) {
  stopifnot(is.data.frame(data))

  if (add_project_kde) {
    data <- add_project_kde_50km(
      data = data,
      lon_var = lon_var,
      lat_var = lat_var,
      project_id_var = project_id_var,
      bandwidth_m = project_kde_bandwidth_m,
      raster_resolution_m = project_kde_resolution_m,
      output_var = "project_kde_50km"
    )
  }

  enduse_vars <- c(
    "enduse.refining",
    "enduse.ammonia",
    "enduse.methanol",
    "enduse.ironsteel",
    "enduse.otherind",
    "enduse.mobility",
    "enduse.power",
    "enduse.gridinj",
    "enduse.chp",
    "enduse.domesticheat",
    "enduse.biofuels",
    "enduse.synfuels",
    "enduse.ch4gridinj",
    "enduse.ch4mobility"
  )

  missing_enduse_vars <- setdiff(enduse_vars, names(data))

  if (length(missing_enduse_vars) > 0) {
    data[missing_enduse_vars] <- 0
  }

  data %>%
    dplyr::mutate(
      dplyr::across(
        dplyr::all_of(enduse_vars),
        ~ tidyr::replace_na(
          suppressWarnings(as.numeric(as.character(.x))),
          0
        )
      )
    ) %>%
    dplyr::mutate(
      any_enduse = as.integer(
        rowSums(
          dplyr::across(dplyr::all_of(enduse_vars)),
          na.rm = TRUE
        ) > 0
      ),
      enduse_industrial = as.integer(
        enduse.refining == 1 |
          enduse.ironsteel == 1 |
          enduse.otherind == 1
      ),
      enduse_chemicals = as.integer(
        enduse.ammonia == 1 |
          enduse.methanol == 1
      ),
      enduse_refining = as.integer(
        enduse.refining == 1
      ),
      enduse_industrial_narrow = as.integer(
        enduse.otherind == 1 |
          enduse.ironsteel == 1
      ),
      enduse_industrial_broad = as.integer(
        enduse.refining == 1 |
          enduse.ironsteel == 1 |
          enduse.otherind == 1 |
          enduse.methanol == 1
      ),
      enduse_industrial_broadest = as.integer(
        enduse.refining == 1 |
          enduse.ironsteel == 1 |
          enduse.otherind == 1 |
          enduse.methanol == 1 |
          enduse.ammonia == 1
      ),
      enduse_transport = as.integer(
        enduse.mobility == 1 |
          enduse.ch4mobility == 1 |
          enduse.synfuels == 1 |
          enduse.biofuels == 1
      ),
      enduse_power = as.integer(
        enduse.power == 1
      ),
      enduse_other = as.integer(
        enduse.gridinj == 1 |
          enduse.ch4gridinj == 1 |
          enduse.domesticheat == 1 |
          enduse.chp == 1
      )
    ) %>%
    dplyr::mutate(
      date_group = dplyr::case_when(
        is.na(date.online) ~ "unknown",
        date.online <= 2025 ~ "early",
        date.online <= 2027 ~ "mid",
        TRUE ~ "late"
      ),
      phase_row = dplyr::case_when(
        enduse.refining == 1 |
          enduse.ammonia == 1 |
          enduse.methanol == 1 |
          enduse.ironsteel == 1 ~ "High",
        enduse.mobility == 1 |
          enduse.synfuels == 1 |
          enduse.ch4mobility == 1 |
          enduse.domesticheat == 1 |
          enduse.chp == 1 |
          enduse.biofuels == 1 |
          enduse.otherind == 1 ~ "Medium",
        enduse.power == 1 |
          enduse.gridinj == 1 |
          enduse.ch4gridinj == 1 ~ "Low",
        TRUE ~ NA_character_
      ),
      market_readiness = assign_market_readiness(phase_row)
    ) %>%
    dplyr::select(-phase_row) %>%
    dplyr::mutate(
      technology = dplyr::case_when(
        technology %in% c("ALK", "PEM") ~ technology,
        TRUE ~ "Other"
      ),
      electricity_source = dplyr::case_when(
        elec.type.dedicated == 1 ~ "Dedicated Renewable",
        !is.na(electricity.type) &
          grepl(
            "dedicated",
            electricity.type,
            ignore.case = TRUE
          ) ~ "Dedicated Renewable",
        TRUE ~ "Other"
      )
    ) %>%
    dplyr::mutate(
      technology = factor(
        technology,
        levels = c("ALK", "Other", "PEM")
      ),
      electricity_source = factor(
        electricity_source,
        levels = c("Other", "Dedicated Renewable")
      ),
      date_group = factor(
        date_group,
        levels = c("early", "mid", "late", "unknown")
      ),
      market_readiness = factor(
        market_readiness,
        levels = c("High", "Medium", "Low", "Unknown")
      )
    )
}


## 7. Build the complete analysis dataset


In [6]:
# ------------------------------------------------------------------------------
# Build the cleaned project-year analysis panel
# ------------------------------------------------------------------------------

build_analysis_dataset <- function(
    final_year = 2026,
    print_missing_diagnostics = TRUE
) {
  external_variables <- read_external_vars()
  data <- read_iea_data()

  non_terminal_levels <- c(
    "Concept",
    "Feasibility study",
    "FID/Construction"
  )

  absorbing_states <- c(
    "Operational",
    "Decommissioned",
    "Cancellation"
  )

  latest_non_missing_numeric <- function(x) {
    x <- as.character(x)
    x <- trimws(x)
    x[x %in% c("", "NA", "NaN", "NULL")] <- NA_character_
    x <- gsub(",", ".", x, fixed = TRUE)
    x <- suppressWarnings(as.numeric(x))
    x <- x[!is.na(x)]

    if (length(x) == 0) {
      return(NA_real_)
    }

    dplyr::last(x)
  }

  clean_status_label <- function(x) {
    x_raw <- trimws(as.character(x))
    x_raw[x_raw == ""] <- NA_character_

    x_clean <- tolower(x_raw)
    x_clean <- gsub("decommision", "decommission", x_clean)
    x_clean <- gsub("cancelled", "cancellation", x_clean)
    x_clean <- gsub("canceled", "cancellation", x_clean)
    x_clean <- gsub("[^a-z0-9]+", " ", x_clean)
    x_clean <- trimws(gsub("\\s+", " ", x_clean))

    dplyr::case_when(
      is.na(x_clean) ~ NA_character_,
      x_clean == "concept" ~ "Concept",
      x_clean %in% c(
        "feasibility",
        "feasibility study"
      ) ~ "Feasibility study",
      x_clean %in% c(
        "fid",
        "fid construction",
        "final investment decision",
        "construction",
        "under construction"
      ) ~ "FID/Construction",
      x_clean %in% c(
        "operational",
        "operation",
        "operating"
      ) ~ "Operational",
      grepl("decommission", x_clean) ~ "Decommissioned",
      x_clean == "dormant" ~ "Dormant",
      x_clean == "on hold" ~ "On hold",
      grepl("cancellation|cancel", x_clean) ~ "Cancellation",
      TRUE ~ NA_character_
    )
  }

  correct_status_history <- function(df) {
    df <- df %>%
      dplyr::arrange(year)

    status_original <- as.character(df$status)
    status_clean <- clean_status_label(status_original)
    status_corrected <- status_clean

    status_harmonized <- !is.na(status_original) &
      !is.na(status_clean) &
      trimws(status_original) != status_clean

    # Sensitivity treatment for the new 2026 IEA statuses Dormant and On hold:
    # keep the project in its most recent prior non-terminal development stage.
    # This is deliberately not treated as progression or failure. If no prior
    # Concept / Feasibility study / FID/Construction stage is observed, the
    # status remains unclassified (NA) rather than being guessed.
    status_prior_stage_carried <- rep(
      FALSE,
      length(status_corrected)
    )
    last_non_terminal_stage <- NA_character_

    for (i in seq_along(status_corrected)) {
      if (status_corrected[i] %in% non_terminal_levels) {
        last_non_terminal_stage <- status_corrected[i]
      } else if (status_corrected[i] %in% c("Dormant", "On hold")) {
        if (!is.na(last_non_terminal_stage)) {
          status_corrected[i] <- last_non_terminal_stage
          status_prior_stage_carried[i] <- TRUE
        } else {
          status_corrected[i] <- NA_character_
        }
      } else if (status_corrected[i] %in% absorbing_states) {
        # Do not carry a pre-terminal development stage across an absorbing state.
        last_non_terminal_stage <- NA_character_
      }
    }

    status_backward_adjusted <- rep(
      FALSE,
      length(status_corrected)
    )

    # Correct implausible backward movement among non-terminal development
    # phases while respecting absorbing-state boundaries.
    ranks <- match(status_corrected, non_terminal_levels)
    future_min_rank <- Inf

    for (i in rev(seq_along(status_corrected))) {
      if (status_corrected[i] %in% absorbing_states) {
        future_min_rank <- Inf
      }

      if (!is.na(ranks[i])) {
        if (ranks[i] > future_min_rank) {
          status_corrected[i] <- non_terminal_levels[future_min_rank]
          status_backward_adjusted[i] <- TRUE
          ranks[i] <- future_min_rank
        } else {
          future_min_rank <- min(future_min_rank, ranks[i])
        }
      }
    }

    status_unknown <- is.na(status_corrected)
    has_prior_known <- cumsum(!status_unknown) > 0
    reverse_known_count <- rev(cumsum(rev(!status_unknown)))
    has_later_known <- reverse_known_count > 0

    status_unknown_violation <- status_unknown &
      has_prior_known &
      has_later_known

    # Once an absorbing state is first observed, propagate it through all
    # subsequent observations. Contradictory non-missing statuses are recorded through
    # absorbing_state_violation.
    first_absorbing_row <- which(
      status_corrected %in% absorbing_states
    )[1]

    absorbing_state_violation <- rep(FALSE, nrow(df))
    absorbing_state_carried <- rep(FALSE, nrow(df))

    if (!is.na(first_absorbing_row)) {
      first_absorbing_state <- status_corrected[first_absorbing_row]
      later_rows <- seq_len(nrow(df)) > first_absorbing_row

      absorbing_state_violation <- later_rows &
        !is.na(status_corrected) &
        status_corrected != first_absorbing_state

      absorbing_state_carried <- later_rows &
        (
          is.na(status_corrected) |
            status_corrected != first_absorbing_state
        )

      status_corrected[later_rows] <- first_absorbing_state
    }

    df$status_original <- status_original
    df$status <- status_corrected
    df$status_harmonized <- status_harmonized
    df$status_prior_stage_carried <- status_prior_stage_carried
    df$status_backward_adjusted <- status_backward_adjusted
    df$status_adjusted <- status_harmonized |
      status_prior_stage_carried |
      status_backward_adjusted |
      absorbing_state_carried
    df$status_unknown <- status_unknown
    df$status_unknown_violation <- status_unknown_violation
    df$absorbing_state_violation <- absorbing_state_violation
    df$status_transition_flag <- status_unknown_violation |
      absorbing_state_violation

    df
  }

  # Project coordinates are treated as time-invariant and set to the latest
  # available valid coordinates for each project.
  data <- data %>%
    dplyr::arrange(reference, year) %>%
    dplyr::group_by(reference) %>%
    dplyr::mutate(
      latitude = latest_non_missing_numeric(latitude),
      longitude = latest_non_missing_numeric(longitude)
    ) %>%
    dplyr::ungroup()

  missing_coords <- data %>%
    dplyr::group_by(reference) %>%
    dplyr::summarise(
      no_lat = all(is.na(latitude)),
      no_lon = all(is.na(longitude)),
      .groups = "drop"
    ) %>%
    dplyr::filter(no_lat | no_lon)

  if (nrow(missing_coords) > 0) {
    message(
      nrow(missing_coords),
      " project(s) have no valid coordinates and will remain NA throughout:\n",
      paste(missing_coords$reference, collapse = ", ")
    )
  }

  # Resolve the DEMO status using online and decommissioning years before the
  # general status-history correction is applied.
  data <- data %>%
    dplyr::mutate(
      status = dplyr::case_when(
        status == "DEMO" &
          !is.na(date.decommissioned) &
          date.decommissioned <= year ~ "Decommissioned",
        status == "DEMO" &
          (is.na(date.decommissioned) |
             date.decommissioned > year) &
          date.online <= year ~ "Operational",
        status == "DEMO" &
          date.online > year ~ "FID/Construction",
        TRUE ~ as.character(status)
      )
    ) %>%
    dplyr::group_by(reference) %>%
    dplyr::group_modify(~ correct_status_history(.x)) %>%
    dplyr::ungroup()

  # Complete each project history through final_year. All project attributes
  # continue to be filled forward. A disappeared non-terminal project becomes
  # a cancellation in the first missing vintage; an absorbing state is
  # propagated instead of being overwritten by disappearance.
  data <- data %>%
    dplyr::group_by(reference) %>%
    dplyr::mutate(
      last_year = max(year),
      last_observed_status = dplyr::last(status[order(year)]),
      cancel_year = dplyr::if_else(
        last_year < final_year &
          !last_observed_status %in% absorbing_states,
        last_year + 1L,
        NA_integer_
      ),
      observed_row = TRUE
    ) %>%
    tidyr::complete(year = seq(min(year), final_year)) %>%
    dplyr::mutate(
      observed_row = tidyr::replace_na(observed_row, FALSE)
    ) %>%
    tidyr::fill(-observed_row, .direction = "down") %>%
    tidyr::fill(
      dplyr::any_of(c("latitude", "longitude")),
      .direction = "downup"
    ) %>%
    dplyr::mutate(
      version = paste0("v", year),
      status_forward_filled = !observed_row &
        (is.na(cancel_year) | year < cancel_year) &
        !is.na(status),
      status_created_cancellation = !is.na(cancel_year) &
        year >= cancel_year,
      status = dplyr::if_else(
        status_created_cancellation,
        "Cancellation",
        status
      )
    ) %>%
    dplyr::mutate(
      dplyr::across(
        dplyr::any_of(c(
          "status_harmonized",
          "status_prior_stage_carried",
          "status_backward_adjusted",
          "status_adjusted",
          "status_unknown",
          "status_unknown_violation",
          "absorbing_state_violation",
          "status_transition_flag"
        )),
        ~ dplyr::if_else(observed_row, .x, FALSE)
      )
    ) %>%
    dplyr::ungroup() %>%
    dplyr::select(
      -last_year,
      -last_observed_status,
      -cancel_year
    )

  # Left truncation marks initial status spells whose start is not observed.
  data <- data %>%
    dplyr::arrange(reference, year) %>%
    dplyr::group_by(reference) %>%
    dplyr::mutate(
      first_year = min(year),
      status_key = dplyr::if_else(
        is.na(status),
        "__missing__",
        status
      ),
      status_change = status_key != dplyr::lag(
        status_key,
        default = dplyr::first(status_key)
      ),
      first_change_year = {
        changed_years <- year[which(status_change & !is.na(year))]
        if (length(changed_years) > 0L) min(changed_years) else NA_integer_
      },
      left_truncated = dplyr::case_when(
        first_year <= 2021 &
          (is.na(first_change_year) |
             year < first_change_year) ~ 1L,
        TRUE ~ 0L
      )
    ) %>%
    dplyr::ungroup() %>%
    dplyr::select(
      -first_year,
      -status_key,
      -status_change,
      -first_change_year
    )

  # Mark all rows belonging to the unfinished final status spell as
  # right-censored.
  data <- data %>%
    dplyr::arrange(reference, year) %>%
    dplyr::group_by(reference) %>%
    dplyr::mutate(
      final_status = status[year == final_year][1],
      is_censored_project = is.na(final_status) |
        !final_status %in% absorbing_states,
      same_as_final = dplyr::if_else(
        is.na(final_status),
        year == final_year,
        !is.na(status) & status == final_status
      ),
      censored_tmp = dplyr::if_else(
        is_censored_project,
        rev(dplyr::cumall(rev(same_as_final))),
        FALSE
      ),
      right_censored = as.integer(censored_tmp)
    ) %>%
    dplyr::ungroup() %>%
    dplyr::select(
      -final_status,
      -is_censored_project,
      -same_as_final,
      -censored_tmp
    )

  data <- data %>%
    dplyr::arrange(reference, year) %>%
    dplyr::group_by(reference) %>%
    dplyr::mutate(
      prev_online = dplyr::lag(date.online),
      delay = as.integer(
        !is.na(prev_online) &
          date.online > prev_online
      ),
      cap_mwel_revision_value = as.character(cap.mwel),
      cap_mwel_revision_value = trimws(cap_mwel_revision_value),
      cap_mwel_revision_value = dplyr::na_if(
        cap_mwel_revision_value,
        ""
      ),
      cap_mwel_revision_value = gsub(
        ",",
        ".",
        cap_mwel_revision_value,
        fixed = TRUE
      ),
      cap_mwel_revision_value = suppressWarnings(
        as.numeric(cap_mwel_revision_value)
      ),
      cap_changed_current = as.integer(
        observed_row &
          !is.na(cap_mwel_revision_value) &
          !is.na(dplyr::lag(cap_mwel_revision_value)) &
          cap_mwel_revision_value != dplyr::lag(cap_mwel_revision_value)
      ),
      instability = dplyr::lag(
        cummax(tidyr::replace_na(cap_changed_current, 0L)),
        default = 0L
      ),
      status_key = dplyr::if_else(
        is.na(status),
        "__missing__",
        status
      ),
      status_group = cumsum(
        dplyr::row_number() == 1 |
          status_key != dplyr::lag(status_key)
      )
    ) %>%
    dplyr::group_by(reference, status_group) %>%
    dplyr::mutate(
      time_in_status = dplyr::row_number() - 1
    ) %>%
    dplyr::ungroup() %>%
    dplyr::select(
      -prev_online,
      -status_key,
      -status_group
    )

  # delay captures upward revisions in the expected online year across
  # successive data vintages, rather than necessarily observed construction
  # delay.

  # Standardise country strings and create multi-country audit variables before
  # joining country-level controls.
  data <- add_country_structure(data, country_var = "country")

  # Country-level controls are averaged equally across all listed ISO3 codes.
  data <- join_external_variables_multicountry(
    data = data,
    external_variables = external_variables,
    country_var = "country"
  )


  data <- prepare_shared_analysis_data(data)

  names(data) <- names(data) %>%
    gsub(" ", "_", .) %>%
    gsub("-", "_", .)

  if (print_missing_diagnostics) {
    missing_pct_by_year <- data %>%
      dplyr::group_by(year) %>%
      dplyr::summarise(
        dplyr::across(
          .cols = dplyr::everything(),
          .fns = ~ round(mean(is.na(.x)) * 100, 2)
        ),
        .groups = "drop"
      ) %>%
      tidyr::pivot_longer(
        cols = -year,
        names_to = "variable",
        values_to = "missing_pct"
      ) %>%
      tidyr::pivot_wider(
        names_from = year,
        values_from = missing_pct
      ) %>%
      dplyr::arrange(variable)

    print(missing_pct_by_year, n = Inf)
  }

  data
}


## 8. Prepare the discrete-time cloglog dataset


In [7]:
# ------------------------------------------------------------------------------
# Prepare the cleaned project-year panel for cloglog models
# ------------------------------------------------------------------------------

prep_cloglog <- function(data) {
  stopifnot(is.data.frame(data))

  extract_year <- function(x) {
    if (
      inherits(x, "Date") |
        inherits(x, "POSIXct") |
        inherits(x, "POSIXlt")
    ) {
      return(as.numeric(format(x, "%Y")))
    }

    if (is.numeric(x)) {
      return(as.numeric(x))
    }

    x <- as.character(x)
    suppressWarnings(as.numeric(substr(x, 1, 4)))
  }

  if (!"date.online" %in% names(data)) {
    data[["date.online"]] <- NA_real_
  }

  absorbing_states <- c(
    "Operational",
    "Decommissioned",
    "Cancellation"
  )

  data %>%
    dplyr::mutate(
      curr_state = dplyr::case_when(
        status == "Concept" ~ 1L,
        status == "Feasibility study" ~ 2L,
        status == "FID/Construction" ~ 3L,
        status == "Operational" ~ 4L,
        status %in% c(
          "Decommissioned",
          "Cancellation"
        ) ~ 0L,
        TRUE ~ NA_integer_
      ),
      curr_absorbing_state = dplyr::case_when(
        status %in% absorbing_states ~ status,
        TRUE ~ NA_character_
      ),
      scheduled_completion_year_current = extract_year(
        .data[["date.online"]]
      )
    ) %>%
    dplyr::arrange(reference, year) %>%
    dplyr::group_by(reference) %>%
    dplyr::mutate(
      prev_state = dplyr::lag(curr_state),
      prev_status = dplyr::lag(status),
      prev_year = dplyr::lag(year),

      # Preserve duration in calendar years before the model-ready copy is
      # standardized below. Prediction notebooks use the raw variable for their
      # x-axis and transform it back to the fitted model scale internally.
      prev_time_in_status_raw = dplyr::lag(time_in_status) + 1L,
      prev_time_in_status = as.numeric(prev_time_in_status_raw),

      prev_scheduled_completion_year = dplyr::lag(
        scheduled_completion_year_current
      ),
      years_to_scheduled_completion =
        prev_scheduled_completion_year - prev_year,
      schedule_revision = as.integer(
        !is.na(scheduled_completion_year_current) &
          !is.na(prev_scheduled_completion_year) &
          scheduled_completion_year_current !=
          prev_scheduled_completion_year
      ),
      previous_schedule_revision_count = cumsum(
        dplyr::lag(schedule_revision, default = 0L)
      ),
      prev_left_truncated = dplyr::lag(
        left_truncated,
        default = dplyr::first(left_truncated)
      ),
      transition_year = year
    ) %>%
    dplyr::ungroup() %>%
    dplyr::mutate(
      progress = as.integer(
        !is.na(prev_state) &
          !is.na(curr_state) &
          prev_state %in% 1:3 &
          curr_state > prev_state
      ),
      failure = as.integer(
        !is.na(prev_state) &
          !is.na(curr_state) &
          prev_state %in% 1:3 &
          curr_state == 0
      ),
      terminal_transition = as.integer(
        !is.na(prev_state) &
          prev_state %in% 1:3 &
          status %in% absorbing_states
      ),
      transition_type = dplyr::case_when(
        progress == 1 &
          status == "Operational" ~ "operationalisation",
        progress == 1 ~ "progress",
        failure == 1 &
          status == "Cancellation" ~ "cancellation",
        failure == 1 &
          status == "Decommissioned" ~ "decommissioning",
        TRUE ~ "none"
      )
    ) %>%
    dplyr::filter(
      !is.na(reference),
      !is.na(country),
      !is.na(prev_state),
      !is.na(curr_state)
    )
}


## 9. Build the full panel and transition dataset


In [8]:
# ------------------------------------------------------------------------------
# Build the full panel and the cloglog analysis dataset
# ------------------------------------------------------------------------------

master_data <- build_analysis_dataset(
  final_year = 2026
)

saveRDS(master_data, "master_data.rds")

cloglog_data <- prep_cloglog(master_data)


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`
• `` -> `...36`
• `` -> `...37`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`
• `` -> `...36`


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
• `` -> `...23`
• `` -> `...24`
• `` -> `...25`
• `` -> `...26`
• `` -> `...27`
• `` -> `...28`
• `` -> `...29`
• `` -> `...30`
• `` -> `...31`
• `` -> `...32`
• `` -> `...33`
• `` -> `...34`
• `` -> `...35`
• `` -> `...36`
• `` -> `...37`
• `` -> `...38`
• `` -> `...39`
• `` -> `...40`
• `` -> `...41`
• `` -> `...42`
• `` -> `...43`
• `` -> `...44`


240 project(s) have no valid coordinates and will remain NA throughout:
1, 1010, 1014, 1018, 1019, 1020, 1024, 1025, 1031, 1040, 1051, 1055, 1075, 1088, 1089, 1092, 1098, 1099, 1101, 1103, 1104, 1105, 1106, 1107, 1111, 1112, 1138, 1150, 1153, 1154, 1168, 1179, 1194, 12, 1203, 1216, 1219, 1225, 1226, 1236, 1238, 1242, 1244, 1246, 1268, 1270, 1278, 1279, 1280, 1284, 1286, 1290, 1296, 1298, 13, 1310, 1313, 1317, 1322, 1324, 1327, 1330, 1341, 1348, 1359, 1363, 1370, 1376, 1407, 1408, 1419, 1423, 1429, 1443, 1445, 1452, 1454, 1455, 1473, 1488, 1490, 1494, 1495, 1496, 1522, 1535, 1625, 1626, 1630, 1645, 1681, 1691, 1721, 1722, 1723, 1727, 1733, 1738, 1747, 178, 1783, 1800, 1815, 1839, 1840, 1848, 1850, 1872, 1882, 1891, 1904, 1905, 1927, 1948, 1991, 1993, 2, 2006, 2025, 2035, 2045, 2069, 2073, 2151, 2247, 2264, 2295, 2321, 2338, 2370, 2380, 2388, 2403, 2404, 2413, 2444, 2451, 2546, 2792, 3184, 3217, 3277, 3317, 3332, 3377, 3381, 3426, 3429, 3461, 35, 361, 3813, 3830, 3834, 39, 3924, 3954, 39

Country codes without external-variable matches: GUF



# A tibble: 98 × 7
   variable                            `2021` `2022` `2023` `2024` `2025` `2026`
   <chr>                                <dbl>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>
 1 .iea_source_excel_row                 0      0      0      0      0      0   
 2 .iea_source_file                      0      0      0      0      0      0   
 3 Anhydrous_Ammonia_Export              0.61   0.54   0.43   0.34   0.3    0.31
 4 Competitive_Industrial_Performance    0.61   0.54   0.43   0.34   0.3    0.31
 5 Country_Complexity_Ranking            0.61   0.54   0.43   0.34   0.3    0.31
 6 Country_Risk_Spread                   0.61   0.54   0.43   0.34   0.3    0.31
 7 Ease_of_doing_business                0.61   0.54   0.43   0.34   0.3    0.31
 8 Freshwater_Withdrawal                 0.61   0.54   0.43   0.34   0.3    0.31
 9 Governance_Score                      0.61   0.54   0.43   0.34   0.3    0.31
10 RISE                                  0.61   0.54   0.43   0.34   0.3    0.31
11 absorb

## 10. Assign broad regions


In [9]:
# ------------------------------------------------------------------------------
# Assign regions, including same-region multi-country projects
# ------------------------------------------------------------------------------

region_levels <- c(
  "Europe",
  "US + Canada",
  "Latin America",
  "Asia Pacific",
  "Africa + Middle East",  
  "Others"
)

add_region_classification <- function(data) {
  data %>%
    dplyr::mutate(
      country = standardize_country_string(country),
      region = assign_region_from_country(country),
      region = factor(
        region,
        levels = region_levels
      )
    )
}

cloglog_data <- add_region_classification(
  cloglog_data
)

master_data <- add_region_classification(
  master_data
)

# Confirm that region was added to both datasets
stopifnot(
  "region" %in% names(cloglog_data),
  "region" %in% names(master_data)
)

# ------------------------------------------------------------------------------
# Classification diagnostics
# ------------------------------------------------------------------------------

cat("\nRegion counts: cloglog_data\n")

cloglog_data %>%
  dplyr::count(
    region,
    .drop = FALSE,
    sort = TRUE
  ) %>%
  print(n = Inf)

cat("\nRegion counts: master_data\n")

master_data %>%
  dplyr::count(
    region,
    .drop = FALSE,
    sort = TRUE
  ) %>%
  print(n = Inf)

cat("\nCountries classified as Others: cloglog_data\n")

cloglog_data %>%
  dplyr::filter(
    region == "Others"
  ) %>%
  dplyr::distinct(
    country
  ) %>%
  dplyr::arrange(
    country
  ) %>%
  print(n = Inf)

cat("\nCountries classified as Others: master_data\n")

master_data %>%
  dplyr::filter(
    region == "Others"
  ) %>%
  dplyr::distinct(
    country
  ) %>%
  dplyr::arrange(
    country
  ) %>%
  print(n = Inf)

cat("\nMulti-country classifications: cloglog_data\n")

cloglog_data %>%
  dplyr::filter(
    multi_country == 1
  ) %>%
  dplyr::distinct(
    country,
    country_primary,
    n_countries,
    region
  ) %>%
  dplyr::arrange(
    region,
    country
  ) %>%
  print(n = Inf)

cat("\nMulti-country classifications: master_data\n")

master_data %>%
  dplyr::filter(
    multi_country == 1
  ) %>%
  dplyr::distinct(
    country,
    country_primary,
    n_countries,
    region
  ) %>%
  dplyr::arrange(
    region,
    country
  ) %>%
  print(n = Inf)



Region counts: cloglog_data


# A tibble: 6 × 2
  region                   n
  <fct>                <int>
1 Europe                5679
2 Asia Pacific          2047
3 US + Canada           1063
4 Latin America          708
5 Africa + Middle East   625
6 Others                   0



Region counts: master_data


# A tibble: 7 × 2
  region                   n
  <fct>                <int>
1 Europe                7433
2 Asia Pacific          2751
3 US + Canada           1403
4 Latin America          972
5 Africa + Middle East   876
6 NA                      43
7 Others                   0



Countries classified as Others: cloglog_data


...

# A tibble: 0 × 1
# ℹ 1 variable: country <chr>



Multi-country classifications: cloglog_data


# A tibble: 5 × 4
  country         country_primary n_countries region
  <chr>           <chr>                 <int> <fct> 
1 DEU DNK         DEU                       2 Europe
2 ESP FRA         ESP                       2 Europe
3 POL CZE SVK HUN POL                       4 Europe
4 PRT ESP         PRT                       2 Europe
5 ROU DEU AUT     ROU                       3 Europe



Multi-country classifications: master_data


# A tibble: 5 × 4
  country         country_primary n_countries region
  <chr>           <chr>                 <int> <fct> 
1 DEU DNK         DEU                       2 Europe
2 ESP FRA         ESP                       2 Europe
3 POL CZE SVK HUN POL                       4 Europe
4 PRT ESP         PRT                       2 Europe
5 ROU DEU AUT     ROU                       3 Europe


In [10]:
# ==============================================================================
# ASSIGN COUNTRY RISK RATING GROUPS
# ==============================================================================

assign_rating_group <- function(
    data,
    risk_var = "Country_Risk_Spread",
    risk_scale = c("proportion", "percent")
) {
  risk_scale <- match.arg(risk_scale)

  if (!risk_var %in% names(data)) {
    stop(
      paste0(
        "The variable '",
        risk_var,
        "' was not found in the supplied data."
      )
    )
  }

  # Thresholds are expressed as proportions:
  # 0.18% = 0.0018, 0.91% = 0.0091, etc.
  # The detailed rating scale continues below CAA through CA and C, so
  # observations with the weakest ratings remain classified rather than being dropped.
  rating_thresholds <- c(
    AAA = 0.0000,
    AA  = 0.0018,
    A   = 0.0091,
    BAA = 0.0207,
    BA  = 0.0324,
    B   = 0.0522,
    CAA = 0.0971,
    CA  = 0.1554,
    C   = 0.2666
  )

  risk_value <- suppressWarnings(
    as.numeric(data[[risk_var]])
  )

  # Convert percentage-point values such as 0.18 into proportions such as 0.0018.
  if (risk_scale == "percent") {
    risk_value <- risk_value / 100
  }

  data %>%
    dplyr::mutate(
      rating_group = dplyr::case_when(
        is.na(risk_value) ~ NA_character_,
        risk_value < rating_thresholds[["AAA"]] ~ NA_character_,
        risk_value < rating_thresholds[["AA"]]  ~ "AAA",
        risk_value < rating_thresholds[["A"]]   ~ "AA",
        risk_value < rating_thresholds[["BAA"]] ~ "A",
        risk_value < rating_thresholds[["BA"]]  ~ "BAA",
        risk_value < rating_thresholds[["B"]]   ~ "BA",
        risk_value < rating_thresholds[["CAA"]] ~ "B",
        risk_value < rating_thresholds[["CA"]]  ~ "CAA",
        risk_value < rating_thresholds[["C"]]   ~ "CA",
        TRUE                                    ~ "C"
      ),
      rating_group = factor(
        rating_group,
        levels = c(
          "AAA",
          "AA",
          "A",
          "BAA",
          "BA",
          "B",
          "CAA",
          "CA",
          "C"
        ),
        ordered = TRUE
      )
    )
}


# ==============================================================================
# APPLY TO BOTH DATASETS
# ==============================================================================

# Use this version when 0.18% is stored as 0.0018.
master_data <- assign_rating_group(
  data = master_data,
  risk_var = "Country_Risk_Spread",
  risk_scale = "proportion"
)

cloglog_data <- assign_rating_group(
  data = cloglog_data,
  risk_var = "Country_Risk_Spread",
  risk_scale = "proportion"
)


# ==============================================================================
# OPTIONAL CHECK
# ==============================================================================

master_data %>%
  dplyr::count(
    rating_group,
    .drop = FALSE
  )

cloglog_data %>%
  dplyr::count(
    rating_group,
    .drop = FALSE
  )


rating_group,n
<ord>,<int>
AAA,4347
AA,3526
A,3036
BAA,1427
BA,609
B,384
CAA,92
CA,6
C,0


rating_group,n
<ord>,<int>
AAA,3359
AA,2669
A,2271
BAA,1052
BA,420
B,280
CAA,62
CA,5
C,0


## 11. Prepare and save the Figure 2 model dataset

The complete transition dataset is saved for audit and alternative analyses. Each transition row represents the interval from the previous project state to the current vintage; project and country covariates are taken from the current project-year record, while state duration and risk-set variables refer to the previous state. The Figure 2 dataset applies the stated analysis filters, constructs `any_enduse`, and standardises continuous variables on the analysis sample.


In [11]:
# ------------------------------------------------------------------------------
# Preserve the complete transition dataset before Figure 2-specific preparation
# ------------------------------------------------------------------------------

saveRDS(
  cloglog_data,
  cloglog_full_path
)

saveRDS(
  master_data,
  "master_data.rds"
)

# ------------------------------------------------------------------------------
# Exclude transitions originating in a left-truncated status spell
# ------------------------------------------------------------------------------
#
# Each row represents the transition from the previous year to the current year.
# The risk set and status duration refer to the previous state. Left truncation
# must therefore be evaluated using prev_left_truncated rather than the
# current-row left_truncated flag. Project and country covariates remain measured
# on the current project-year record.

left_truncation_audit <- tibble::tibble(
  rows_before_filter = nrow(cloglog_data),
  rows_excluded = sum(
    cloglog_data$prev_left_truncated != 0,
    na.rm = TRUE
  ),
  rows_retained = sum(
    cloglog_data$prev_left_truncated == 0,
    na.rm = TRUE
  )
)

print(
  left_truncation_audit
)

cloglog_data <- cloglog_data %>%
  dplyr::filter(
    prev_left_truncated == 0
  )


# ------------------------------------------------------------------------------
# Signed log1p transformation
# ------------------------------------------------------------------------------
#
# sign(x) * log(1 + abs(x)) is defined for positive, zero, and negative values.
# For non-negative variables, it is identical to log1p(x).

signed_log1p <- function(x) {
  if (!is.numeric(x)) {
    stop("signed_log1p() requires a numeric vector.")
  }

  sign(x) * log1p(abs(x))
}

log_transform_vars <- c(
  "cap.mwel",
  "Anhydrous_Ammonia_Export",
  "Country_Risk_Spread",
  "Freshwater_Withdrawal"
)

cloglog_data <- cloglog_data %>%
  dplyr::mutate(
    dplyr::across(
      dplyr::any_of(log_transform_vars),
      signed_log1p,
      .names = "{.col}_log"
    )
  )


# ------------------------------------------------------------------------------
# Treat region as a standard categorical control
# ------------------------------------------------------------------------------

if ("region" %in% names(cloglog_data)) {
  cloglog_data <- cloglog_data %>%
    dplyr::mutate(
      region = factor(region)
    )
}


# ------------------------------------------------------------------------------
# Standardize selected numeric variables
# ------------------------------------------------------------------------------
#
# prev_time_in_status_raw records time in stage in calendar years. Only the model
# covariate prev_time_in_status is standardized.

numeric_vars <- c(
  "cap.mwel_log",
  "prev_time_in_status",
  "Anhydrous_Ammonia_Export_log",
  "Competitive_Industrial_Performance",
  "Country_Complexity_Ranking",
  "Ease_of_doing_business",
  "Freshwater_Withdrawal_log",
  "Governance_Score",
  "Country_Risk_Spread_log",
  "RISE",
  "project_kde_50km"
)

available_numeric_vars <- intersect(
  numeric_vars,
  names(cloglog_data)
)

# Store the exact mean and SD used for every standardized model covariate.
# These values are required to translate fitted coefficients back to changes
# in the original variables.
standardization_parameters <- tibble::tibble(
  variable = available_numeric_vars,
  mean = vapply(
    available_numeric_vars,
    function(variable_name) {
      mean(
        cloglog_data[[variable_name]],
        na.rm = TRUE
      )
    },
    numeric(1)
  ),
  sd = vapply(
    available_numeric_vars,
    function(variable_name) {
      stats::sd(
        cloglog_data[[variable_name]],
        na.rm = TRUE
      )
    },
    numeric(1)
  )
)

if (standardize_numeric) {
  for (variable_name in available_numeric_vars) {
    parameter_row <- standardization_parameters[
      standardization_parameters$variable == variable_name,
    ]

    variable_mean <- parameter_row$mean
    variable_sd <- parameter_row$sd

    if (
      length(variable_mean) == 1 &&
      length(variable_sd) == 1 &&
      is.finite(variable_mean) &&
      is.finite(variable_sd) &&
      variable_sd > 0
    ) {
      cloglog_data[[variable_name]] <- (
        cloglog_data[[variable_name]] - variable_mean
      ) / variable_sd
    }
  }
}


# ------------------------------------------------------------------------------
# Construct the aggregate any-end-use indicator
# ------------------------------------------------------------------------------

enduse_controls_grouped <- c(
  "enduse_industrial",
  "enduse_transport",
  "enduse_chemicals",
  "enduse_power",
  "enduse_other"
)

enduse_controls_grouped <- enduse_controls_grouped[
  enduse_controls_grouped %in% names(cloglog_data)
]

if (length(enduse_controls_grouped) == 0) {
  stop("No grouped end-use variables were found in cloglog_data.")
}

enduse_matrix <- cloglog_data %>%
  dplyr::select(
    dplyr::all_of(enduse_controls_grouped)
  ) %>%
  dplyr::mutate(
    dplyr::across(
      dplyr::everything(),
      as.numeric
    )
  )

cloglog_data$any_enduse <- dplyr::case_when(
  rowSums(!is.na(enduse_matrix)) == 0 ~ NA_integer_,
  rowSums(enduse_matrix, na.rm = TRUE) > 0 ~ 1L,
  TRUE ~ 0L
)


# ------------------------------------------------------------------------------
# Save the model-ready dataset consumed by the Figure 2 notebook
# ------------------------------------------------------------------------------

attr(
  cloglog_data,
  "standardize_numeric"
) <- standardize_numeric

attr(
  cloglog_data,
  "standardization_parameters"
) <- standardization_parameters

utils::write.csv(
  standardization_parameters,
  "cloglog_standardization_parameters.csv",
  row.names = FALSE
)

saveRDS(
  cloglog_data,
  cloglog_figure2_path
)

# Export a complete raw -> panel -> transition -> eligible-sample reconciliation.
# All summaries use the same identifier/year validity rule as Figure 1 / ST1-2.
sample_count_audits <- list(
  raw = raw_sample_count_audit,
  panel = audit_project_year_keys(master_data, "Reconstructed panel"),
  transitions = audit_project_year_keys(readRDS(cloglog_full_path), "Full transition dataset"),
  eligible = audit_project_year_keys(cloglog_data, "After origin-spell left-truncation exclusion")
)
for (audit_name in names(sample_count_audits)) {
  write_project_year_audit(sample_count_audits[[audit_name]], paste0("sample_count_", audit_name))
}
sample_count_audit <- list(
  schema_version = 1L,
  source_md5 = tools::md5sum(c("master_data.rds", cloglog_full_path, cloglog_figure2_path)),
  summary = dplyr::bind_rows(lapply(sample_count_audits, function(x) x$summary)),
  audits = sample_count_audits,
  note = paste(
    "Rows are audited without changing the analysis sample.",
    "Missing reference, empty reference and missing/invalid year exclusions are mutually exclusive, in that order.",
    "Duplicate-key counts refer only to valid keys; extra rows exclude the first occurrence.",
    "Observed/reconstructed row flags in transition datasets describe the destination-vintage record.",
    "Full transition data apply prep_cloglog's valid-state/country restrictions before the left-truncation exclusion.",
    "Eligible data precede stage-specific restrictions and model complete-case selection.",
    "Workbook names and original Excel row numbers are in the flagged-row exports.",
    "On reconstructed rows, workbook provenance is carried from the source record; observed_row identifies these rows."
  )
)
saveRDS(sample_count_audit, "sample_count_audit.rds")
utils::write.csv(sample_count_audit$summary, "sample_count_reconciliation.csv", row.names = FALSE)
print(sample_count_audit$summary, n = Inf, width = Inf)
message(sample_count_audit$note)



# A tibble: 1 × 3
  rows_before_filter rows_excluded rows_retained
               <int>         <int>         <int>
1              10122          4207          5915


# A tibble: 1 × 16
  dataset       total_rows missing_reference_rows empty_reference_rows
  <chr>              <int>                  <int>                <int>
1 Raw IEA input      12478                     30                    0
  missing_year_rows_with_valid_reference valid_key_rows
                                   <int>          <int>
1                                      0          12448
  distinct_valid_project_years valid_projects duplicate_keys
                         <int>          <int>          <int>
1                        12448           3200              0
  extra_duplicate_rows conflicting_duplicate_keys whitespace_only_reference_rows
                 <int>                      <int>                          <int>
1                    0                          0                              0
  observed_rows reconstructed_rows project_years_with_observed_record
          <int>              <int>                              <int>
1         12478                  0

# A tibble: 4 × 16
  dataset                                      total_rows missing_reference_rows
  <chr>                                             <int>                  <int>
1 Raw IEA input                                     12478                     30
2 Reconstructed panel                               13478                     30
3 Full transition dataset                           10122                      0
4 After origin-spell left-truncation exclusion       5915                      0
  empty_reference_rows missing_year_rows_with_valid_reference valid_key_rows
                 <int>                                  <int>          <int>
1                    0                                      0          12448
2                    0                                      0          13448
3                    0                                      0          10122
4                    0                                      0           5915
  distinct_valid_project_years va

Rows are audited without changing the analysis sample. Missing reference, empty reference and missing/invalid year exclusions are mutually exclusive, in that order. Duplicate-key counts refer only to valid keys; extra rows exclude the first occurrence. Observed/reconstructed row flags in transition datasets describe the destination-vintage record. Full transition data apply prep_cloglog's valid-state/country restrictions before the left-truncation exclusion. Eligible data precede stage-specific restrictions and model complete-case selection. Workbook names and original Excel row numbers are in the flagged-row exports. On reconstructed rows, workbook provenance is carried from the source record; observed_row identifies these rows.

